# 06. H3 Next-Hour ONNX and Go Deployment Validation  
# 06. H3 下一小时 ONNX 与 Go 部署验证

**Pipeline position / 主线位置:** selected next-hour model from Notebook 02 -> ONNX contract -> Python and Go parity.  
**流程位置：** Notebook 02 选出的下一小时模型 -> ONNX 部署约定 -> Python 与 Go 一致性验证。

This notebook checks the deployable model package. Python and Go read the same ONNX file, feature order, preprocessing medians, and fixed test cases. The parity test confirms that both runtimes return the same prediction; it is separate from the model-accuracy test.  
本 Notebook 检查可部署模型包。Python 和 Go 读取同一个 ONNX 文件、特征顺序、预处理中位数和固定测试案例。一致性测试确认两个运行环境给出相同预测，它与模型准确率测试是两件事。

**Inputs / 输入:** files in `model_h3_next_hour_onnx_go_deployment`.  
**Outputs / 输出:** Python ONNX validation and Go parity result.

## 1. Environment and paths / 环境与路径

Set `H3_ONNX_GO_PROJECT_DIR` when Jupyter does not start in the project directory. Retraining is disabled by default because it requires the upstream hourly cache.  
当 Jupyter 启动目录不是项目目录时，设置 `H3_ONNX_GO_PROJECT_DIR`。默认不重新训练，因为训练需要上游小时级缓存。

In [ ]:
# Cell 1 - Imports and portable configuration / 导入包与可移植配置
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_ROOT = Path(
    os.getenv("CHICAGO_TNP_REPO_DIR", str(Path.cwd()))
).expanduser().resolve()
default_package = REPO_ROOT / "model_h3_next_hour_onnx_go_deployment"
PROJECT_DIR = Path(
    os.getenv(
        "H3_ONNX_GO_PROJECT_DIR",
        str(REPO_ROOT if REPO_ROOT.name == "model_h3_next_hour_onnx_go_deployment" else default_package),
    )
).expanduser().resolve()
MODEL_DIR = PROJECT_DIR / "model"
TESTDATA_DIR = PROJECT_DIR / "testdata"

cache_value = os.getenv("CHICAGO_TNP_HOURLY_CACHE")
HOURLY_CACHE = Path(cache_value).expanduser().resolve() if cache_value else None
RUN_TRAINING = False

print("PROJECT_DIR / 项目目录:", PROJECT_DIR)
print("Model exists / 模型存在:", (MODEL_DIR / "h3_next_hour_pickups.onnx").exists())
print("Training enabled / 是否重新训练:", RUN_TRAINING)
print("Hourly cache / 小时缓存:", HOURLY_CACHE)

## 2. Inspect the deployment contract / 查看部署约定

The schema fixes the 61-feature order, training medians, valid H3 codes, ONNX tensor names, and nonnegative postprocessing. Python and Go both use this file.  
schema 固定 61 个特征的顺序、训练期中位数、合法 H3 编码、ONNX 张量名称和非负后处理。Python 与 Go 共用该文件。

In [ ]:
# Cell 2 - Inspect schema and saved metrics / 查看 schema 与保存指标
schema = json.loads((MODEL_DIR / "feature_schema.json").read_text(encoding="utf-8"))
metrics = json.loads((MODEL_DIR / "model_metrics.json").read_text(encoding="utf-8"))

print("Feature count / 特征数量:", len(schema["feature_order"]))
print("Input tensor / 输入张量:", schema["input_name"], schema["input_shape"])
print("Output tensor / 输出张量:", schema["output_name"])
print("Model metrics / 模型指标:", metrics["model"])
print("Current-hour baseline / 当前小时基线:", metrics["naive_current_hour"])
print("Last-week baseline / 上周同期基线:", metrics["naive_last_week"])

## 3. Optional retraining and ONNX export / 可选的重新训练与 ONNX 导出

Training uses 2022-2023 and evaluates 2024. The command regenerates the model, schema, metrics, and fixed cross-language cases.  
训练使用 2022-2023，评估使用 2024。该命令会重新生成模型、schema、指标和固定跨语言样本。

In [ ]:
# Cell 3 - Optional training run / 可选训练运行
if RUN_TRAINING:
    if HOURLY_CACHE is None or not HOURLY_CACHE.exists():
        raise FileNotFoundError(
            "Set CHICAGO_TNP_HOURLY_CACHE to hourly_complete_grid.pkl. / "
            "请将 CHICAGO_TNP_HOURLY_CACHE 设置为 hourly_complete_grid.pkl。"
        )
    command = [
        sys.executable,
        str(PROJECT_DIR / "python" / "train_export.py"),
        "--input", str(HOURLY_CACHE),
        "--output", str(MODEL_DIR),
        "--cases-output", str(TESTDATA_DIR / "parity_test_cases.json"),
    ]
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
else:
    print("Using included validated model artifacts. / 使用随包的已验证模型产物。")

## 4. Python ONNX Runtime validation / Python ONNX Runtime 验证

This cell loads the exported ONNX model directly and compares its predictions with fixed Python references. It does not call the original scikit-learn estimator.  
本 cell 直接加载导出的 ONNX 模型，并与固定 Python 参考结果比较，不调用原始 scikit-learn 模型。

In [ ]:
# Cell 4 - Validate ONNX in Python / 在 Python 中验证 ONNX
python_validation = [
    sys.executable,
    str(PROJECT_DIR / "python" / "verify_onnx.py"),
    "--model", str(MODEL_DIR / "h3_next_hour_pickups.onnx"),
    "--schema", str(MODEL_DIR / "feature_schema.json"),
    "--cases", str(TESTDATA_DIR / "parity_test_cases.json"),
]
subprocess.run(python_validation, cwd=PROJECT_DIR, check=True)

## 5. Go cross-language validation / Go 跨语言验证

The Go program compiles from source, loads the same ONNX model, and checks the same twenty cases. `PASS` confirms that Go and Python ONNX Runtime agree within 0.001 pickups.  
Go 程序从源码编译，加载同一个 ONNX 模型并检查相同的 20 条案例。出现 `PASS` 表示 Go 与 Python ONNX Runtime 在 0.001 个 pickup 的容限内一致。

In [ ]:
# Cell 5 - Run Go parity validation / 运行 Go 一致性验证
if shutil.which("go"):
    subprocess.run([str(PROJECT_DIR / "run_go_test.sh")], cwd=PROJECT_DIR, check=True)
else:
    print("Go is not installed; see README.md. / 尚未安装 Go，请查看 README.md。")

## 6. Result interpretation / 结果解读

The included model reaches MAE 12.66 and R² 0.9412 on the 2024 test period, outperforming current-hour and last-week baselines. The parity test verifies deployment correctness rather than model accuracy: it confirms that Go executes the exported Python model consistently.  
随包模型在 2024 测试期达到 MAE 12.66 和 R² 0.9412，优于当前小时与上周同期基线。一致性测试验证的是部署正确性，而不是模型准确率：它确认 Go 能够稳定执行 Python 导出的同一模型。

## Result snapshot / 结果快照

- The included ONNX model has 61 ordered features and reaches MAE 12.66 and R² 0.9412 on 2024. / 随包 ONNX 模型包含 61 个有固定顺序的特征，在 2024 测试期达到 MAE 12.66、R² 0.9412。
- Python ONNX Runtime and Go must agree within 0.001 pickups on the fixed parity cases. / Python ONNX Runtime 与 Go 在固定一致性案例上的误差必须小于 0.001 个 pickup。
- A parity `PASS` proves the package is portable; it does not replace the accuracy metrics above. / 一致性测试显示 `PASS` 证明模型可移植，但不能替代上面的准确率指标。

**Next / 下一步:** Notebook 07 applies the hourly flow framework to a real event-detection case.